# Synthetic 3D T1 Brain Tumor MRI (NIfTI Latent Diffusion)

This notebook trains an unconditional 3D latent diffusion model for synthetic T1 brain tumor MRI volumes directly from NIfTI data.

## Workflow (NIfTI 3D, Unconditional T1 Generation)
1. Discover 3D T1 `.nii`/`.nii.gz` files recursively.
2. Train a 3D `AutoencoderKL` on T1 volumes.
3. Freeze the autoencoder and train a latent DDPM UNet.
4. Use EMA UNet for stable sampling and decode synthetic T1 volumes.
5. Save generated volumes as NIfTI with optional reference affine/header.

In [6]:
# Install once in the CURRENT notebook kernel, then restart kernel and run all.
# In Jupyter/VS Code, prefer %pip (not !pip).
%pip install monai monai-generative nibabel scikit-image tqdm matplotlib

import importlib
import os
import re
import copy
import random
from pathlib import Path

required_modules = ["nibabel", "skimage", "monai", "generative"]
missing = [m for m in required_modules if importlib.util.find_spec(m) is None]
if missing:
    raise ModuleNotFoundError(
        f"Missing modules: {missing}. Run the %pip install cell above and restart kernel."
    )

import numpy as np
import nibabel as nib
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torch.cuda.amp import autocast, GradScaler
from torch.optim.lr_scheduler import CosineAnnealingLR

from tqdm.auto import tqdm

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    CropForegroundd,
    SpatialPadd,
    CenterSpatialCropd,
    ScaleIntensityRangePercentilesd,
    RandFlipd,
    RandRotate90d,
    RandAffined,
    EnsureTyped,
)
from monai.data import Dataset
from monai.utils import set_determinism

from generative.networks.nets import AutoencoderKL, DiffusionModelUNet
from generative.networks.schedulers import DDPMScheduler

print("All required packages imported successfully.")

All required packages imported successfully.


In [7]:
# =========================
# Config (follow Updated_latent_diff style: mount drive + folder link)
# =========================
DATASET_ROOT_OVERRIDE = None  # optional hard override path
DRIVE_FOLDER_LINK = "https://drive.google.com/drive/folders/1gjw-p34h9s_ij4reC53nNgovyIM1KN16?usp=drive_link"
MYDRIVE_DATASET_ROOT = Path("/content/drive/MyDrive/Dataset")
LINK_DOWNLOAD_ROOT = Path("/content/dataset_from_link")

AUTO_MOUNT_DRIVE = True
AUTO_DOWNLOAD_FROM_LINK = True
AUTO_EXTRACT_ZIPS = True

import shutil
import zipfile
import subprocess
import sys

IS_COLAB = (
    "COLAB_GPU" in os.environ
    or "COLAB_RELEASE_TAG" in os.environ
    or Path.cwd().as_posix().startswith("/content")
)


def _norm_name(text: str) -> str:
    return re.sub(r"[^a-z0-9]", "", text.lower())


def _unique_paths(paths):
    seen = set()
    out = []
    for p in paths:
        key = str(p)
        if key not in seen:
            seen.add(key)
            out.append(p)
    return out


def _ensure_colab_drive_mounted():
    if not IS_COLAB:
        raise RuntimeError(
            "This notebook is configured for Colab runtime. "
            "Run it in Colab so /content/drive can be mounted."
        )

    if not AUTO_MOUNT_DRIVE:
        return

    if Path("/content/drive/MyDrive").exists():
        return

    from google.colab import drive
    print("Mounting Google Drive at /content/drive ...")
    drive.mount("/content/drive")


def _maybe_download_from_folder_link(folder_link: str, download_root: Path):
    if not AUTO_DOWNLOAD_FROM_LINK or not folder_link:
        return None

    # Reuse prior download to avoid pulling the folder repeatedly.
    if download_root.exists() and any(download_root.iterdir()):
        print(f"Using cached folder-link download at {download_root}")
        return download_root

    download_root.mkdir(parents=True, exist_ok=True)

    try:
        import gdown  # type: ignore
    except Exception:
        print("Installing gdown ...")
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown  # type: ignore

    print("Downloading shared Drive folder via link (this may take a while) ...")
    try:
        files = gdown.download_folder(
            url=folder_link,
            output=str(download_root),
            quiet=False,
            remaining_ok=True,
        )
        if files:
            print(f"Downloaded {len(files)} file(s) from folder link.")
        else:
            print("No files returned by folder-link download.")
    except Exception as e:
        raise RuntimeError(
            "Failed to download the Drive folder link. "
            "Check link permissions (Anyone with link can view)."
        ) from e

    return download_root


def _find_first_dir_by_norm(root: Path, target_norm: str, recursive: bool = False):
    if not root.exists() or not root.is_dir():
        return None

    iterator = root.rglob("*") if recursive else root.iterdir()
    for p in iterator:
        if p.is_dir() and _norm_name(p.name) == target_norm:
            return p
    return None


def _find_split_dir_by_token(dataset_root: Path, split: str, recursive: bool = False):
    if not dataset_root.exists() or not dataset_root.is_dir():
        return None

    token = "training" if split == "train" else "validation"
    iterator = dataset_root.rglob("*") if recursive else dataset_root.iterdir()

    best = None
    best_key = None

    for p in iterator:
        if not p.is_dir():
            continue

        try:
            rel_depth = len(p.relative_to(dataset_root).parts)
        except Exception:
            continue

        if recursive and rel_depth > 3:
            continue

        n = _norm_name(p.name)
        if token not in n:
            continue

        # Prefer shallow, dataset-like names and avoid case-level folders ending with many digits.
        penalty = 0
        if "data" in n:
            penalty -= 2
        if "nifti" in n:
            penalty -= 1
        if re.search(r"\d{3,}$", n):
            penalty += 2

        key = (rel_depth, penalty, len(p.name))
        if best is None or key < best_key:
            best = p
            best_key = key

    return best


def _find_zip_for_split(dataset_root: Path, split: str):
    split_token = "training" if split == "train" else "validation"
    zip_files = sorted(dataset_root.glob("*.zip"))

    # Prefer names containing both split token and t1.
    for z in zip_files:
        n = z.name.lower()
        if split_token in n and "t1" in n:
            return z

    # Fallback to split token only.
    for z in zip_files:
        n = z.name.lower()
        if split_token in n:
            return z

    return None


def _extract_zip(zip_path: Path, dataset_root: Path):
    print(f"Extracting {zip_path.name} into {dataset_root} ...")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(dataset_root)


def _ensure_split_dir(dataset_root: Path, canonical_name: str, split: str):
    target = dataset_root / canonical_name
    target_norm = _norm_name(canonical_name)

    if target.exists() and target.is_dir():
        return target

    # 1) Exact canonical match by normalized name.
    found = _find_first_dir_by_norm(dataset_root, target_norm, recursive=False)
    if found is None:
        found = _find_first_dir_by_norm(dataset_root, target_norm, recursive=True)

    # 2) Heuristic split detection (e.g. MICCAI_BraTS2020_TrainingData_nifti).
    if found is None:
        found = _find_split_dir_by_token(dataset_root, split=split, recursive=False)
    if found is None:
        found = _find_split_dir_by_token(dataset_root, split=split, recursive=True)

    # 3) If still missing, extract matching zip and retry detection.
    if found is None and AUTO_EXTRACT_ZIPS:
        z = _find_zip_for_split(dataset_root, split=split)
        if z is not None:
            _extract_zip(z, dataset_root)
            found = _find_first_dir_by_norm(dataset_root, target_norm, recursive=True)
            if found is None:
                found = _find_split_dir_by_token(dataset_root, split=split, recursive=True)

    if found is not None:
        if found != target and not target.exists():
            print(f"Moving split folder to canonical path: {found} -> {target}")
            shutil.move(str(found), str(target))
        return target if target.exists() else found

    return None


def _expand_dataset_candidates(base_root: Path):
    cands = [base_root, base_root / "Dataset"]
    # Also catch nested folder named Dataset from link downloads.
    if base_root.exists() and base_root.is_dir():
        for p in base_root.rglob("*"):
            if p.is_dir() and _norm_name(p.name) == "dataset":
                rel_depth = len(p.relative_to(base_root).parts)
                if rel_depth <= 3:
                    cands.append(p)
    return _unique_paths(cands)


def _prepare_dataset_root(dataset_root: Path):
    if not dataset_root.exists() or not dataset_root.is_dir():
        return None

    print(f"Checking dataset root candidate: {dataset_root}")
    zip_files = sorted(dataset_root.glob("*.zip"))
    if zip_files:
        print("Zip files detected:")
        for z in zip_files:
            print(" -", z.name)

    train_dir = _ensure_split_dir(dataset_root, "T1_Training", split="train")
    val_dir = _ensure_split_dir(dataset_root, "T1_Validation", split="val")

    if train_dir is not None and val_dir is not None:
        return dataset_root.resolve()
    return None


def _discover_dataset_root() -> Path:
    _ensure_colab_drive_mounted()

    candidate_roots = []
    if DATASET_ROOT_OVERRIDE:
        candidate_roots.append(Path(DATASET_ROOT_OVERRIDE).expanduser())

    candidate_roots.append(MYDRIVE_DATASET_ROOT)

    downloaded = _maybe_download_from_folder_link(DRIVE_FOLDER_LINK, LINK_DOWNLOAD_ROOT)
    if downloaded is not None:
        candidate_roots.append(downloaded)

    checked = []
    for base in _unique_paths(candidate_roots):
        for cand in _expand_dataset_candidates(base):
            checked.append(cand)
            prepared = _prepare_dataset_root(cand)
            if prepared is not None:
                return prepared

    checked_text = "\n".join(str(p) for p in _unique_paths(checked))
    raise FileNotFoundError(
        "Could not find/prepare dataset root containing both T1_Training and T1_Validation.\n"
        f"Checked:\n{checked_text}\n\n"
        "Expected either:\n"
        "1) /content/drive/MyDrive/Dataset with T1_Training.zip and T1_Validation.zip, or\n"
        "2) Accessible shared folder link that contains those files."
    )


DATASET_ROOT = _discover_dataset_root()
TRAIN_ROOT = DATASET_ROOT / "T1_Training"
VAL_ROOT = DATASET_ROOT / "T1_Validation"
PROJECT_ROOT = DATASET_ROOT.parent
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "synthetic_t1_3d_ldm"

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
if not TRAIN_ROOT.exists():
    raise FileNotFoundError(f"Training folder not found: {TRAIN_ROOT}")
if not VAL_ROOT.exists():
    raise FileNotFoundError(f"Validation folder not found: {VAL_ROOT}")

# Reproducibility
SEED = 42
set_determinism(seed=SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Data
T1_FILENAME_FILTER = "t1"   # use None to keep all nii files
NUM_WORKERS = 0
BATCH_SIZE = 1
GRAD_ACCUM_STEPS = 2

# Spatial setup (H, W, D)
TARGET_SHAPE = (160, 128, 128)
TARGET_PIXDIM = (1.0, 1.0, 1.0)

# Training
AE_EPOCHS = 100
DIFF_EPOCHS = 220
AE_LR = 1e-4
DIFF_LR = 8e-5
NUM_DIFFUSION_TIMESTEPS = 1000
INFERENCE_STEPS = 300
EMA_DECAY = 0.999
OFFSET_NOISE = 0.1
SAVE_EVERY = 5

# Intensity normalization percentiles
NORM_LOW_PCT = 0.5
NORM_HIGH_PCT = 99.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
use_amp = device.type == "cuda"
print("Using device:", device)
print("AMP enabled:", use_amp)
print("Working dir:", Path.cwd())
print("Running in Colab-like runtime:", IS_COLAB)
print("Dataset root:", DATASET_ROOT)
print("Training root:", TRAIN_ROOT)
print("Validation root:", VAL_ROOT)
print("Output root:", OUTPUT_ROOT)

Using cached folder-link download at /content/dataset_from_link
Checking dataset root candidate: /content/dataset_from_link
Zip files detected:
 - T1_Training.zip
 - T1_Validation.zip
Using device: cuda
AMP enabled: True
Working dir: /content
Running in Colab-like runtime: True
Dataset root: /content/dataset_from_link
Training root: /content/dataset_from_link/T1_Training
Validation root: /content/dataset_from_link/T1_Validation
Output root: /content/outputs/synthetic_t1_3d_ldm


In [ ]:
# =========================
# OOM guard (adaptive device + safe defaults)
# =========================
import gc
import os

# Helps allocator handle large 3D tensors with fewer fragmentation issues
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Conservative defaults for 3D stability
TARGET_SHAPE = (96, 96, 64)
BATCH_SIZE = 1
NUM_WORKERS = 0
SAVE_EVERY = 1

# Requested training lengths
AE_EPOCHS = 100
DIFF_EPOCHS = 100

# Keep CPU path enabled so AE can still run (very slow) when CUDA is unavailable.
CPU_FAST_MODE = True

# Optional smoke-test switch. Leave False for full training lengths above.
SMOKE_TEST_MODE = False
if SMOKE_TEST_MODE:
    AE_EPOCHS = min(AE_EPOCHS, 1)
    DIFF_EPOCHS = min(DIFF_EPOCHS, 1)
    INFERENCE_STEPS = min(globals().get("INFERENCE_STEPS", 300), 80)

# Runtime-adaptive device selection
min_free_gb_for_cuda = 2.0
selected_device = torch.device("cpu")
cuda_free_gb = 0.0
cuda_total_gb = 0.0
cuda_probe_ok = False
cuda_fallback_reason = ""

if torch.cuda.is_available():
    try:
        free_b, total_b = torch.cuda.mem_get_info()
        cuda_free_gb = free_b / (1024 ** 3)
        cuda_total_gb = total_b / (1024 ** 3)

        try:
            _probe = torch.zeros((1,), device="cuda")
            del _probe
            cuda_probe_ok = True
        except Exception as e:
            cuda_probe_ok = False
            cuda_fallback_reason = f"CUDA probe allocation failed: {repr(e)}"

        if cuda_probe_ok and cuda_free_gb >= min_free_gb_for_cuda:
            selected_device = torch.device("cuda")
        elif cuda_probe_ok:
            cuda_fallback_reason = (
                f"Insufficient free VRAM ({cuda_free_gb:.2f} GB < {min_free_gb_for_cuda:.2f} GB threshold)."
            )
    except Exception as e:
        # If mem_get_info fails, keep CPU fallback and surface the root cause.
        selected_device = torch.device("cpu")
        cuda_fallback_reason = f"CUDA memory query failed: {repr(e)}"
else:
    cuda_fallback_reason = "torch.cuda.is_available() returned False."

device = selected_device
use_amp = device.type == "cuda"

if device.type == "cuda":
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

print("OOM guard applied:")
print(" - TARGET_SHAPE:", TARGET_SHAPE)
print(" - BATCH_SIZE:", BATCH_SIZE)
print(" - NUM_WORKERS:", NUM_WORKERS)
print(" - SAVE_EVERY:", SAVE_EVERY)
print(" - CPU_FAST_MODE:", CPU_FAST_MODE)
print(" - SMOKE_TEST_MODE:", SMOKE_TEST_MODE)
print(" - device:", device)
print(" - torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    print(f" - cuda free/total GB: {cuda_free_gb:.2f}/{cuda_total_gb:.2f}")
    print(" - cuda probe allocation ok:", cuda_probe_ok)
if device.type != "cuda":
    print(" - cuda fallback reason:", cuda_fallback_reason)
print(" - AE_EPOCHS:", AE_EPOCHS)
print(" - DIFF_EPOCHS:", DIFF_EPOCHS)
print(" - INFERENCE_STEPS:", INFERENCE_STEPS)

OOM guard applied:
 - TARGET_SHAPE: (96, 96, 64)
 - BATCH_SIZE: 1
 - NUM_WORKERS: 0
 - SAVE_EVERY: 1
 - CPU_FAST_MODE: True
 - device: cpu
 - cuda free/total GB: 0.00/0.00
 - AE_EPOCHS: 1
 - DIFF_EPOCHS: 0
 - INFERENCE_STEPS: 10


In [9]:
# Runtime diagnostics (CUDA + skip behavior)
print("torch.cuda.is_available():", torch.cuda.is_available())
if torch.cuda.is_available():
    try:
        free_b, total_b = torch.cuda.mem_get_info()
        print("cuda mem_get_info free/total GB:", free_b / (1024**3), total_b / (1024**3))
    except Exception as e:
        print("cuda mem_get_info error:", repr(e))
    try:
        _t = torch.zeros((1,), device="cuda")
        del _t
        print("cuda tiny allocation: OK")
    except Exception as e:
        print("cuda tiny allocation error:", repr(e))
print("device selected by OOM guard:", device)
print("AE_EPOCHS, DIFF_EPOCHS:", AE_EPOCHS, DIFF_EPOCHS)

torch.cuda.is_available(): True
cuda mem_get_info error: AcceleratorError("CUDA error: out of memory\nSearch for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n")
cuda tiny allocation error: AcceleratorError("CUDA error: out of memory\nSearch for `cudaErrorMemoryAllocation' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.\nCUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.\nFor debugging consider passing CUDA_LAUNCH_BLOCKING=1\nCompile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.\n")
device selected by OOM guard: cpu
AE_EPOCHS, DIFF_EPOCH

In [10]:
# Quick dataset path diagnostic (safe to run multiple times)
from pathlib import Path
import os

def _find_dataset_roots(search_roots, max_depth=7, max_dirs=40000, max_hits=20):
    hits = []
    for root in search_roots:
        root = Path(root).expanduser()
        if not root.exists() or not root.is_dir():
            continue
        base_depth = len(root.parts)
        scanned = 0
        for dirpath, dirnames, _ in os.walk(root):
            scanned += 1
            if scanned > max_dirs:
                break
            cur = Path(dirpath)
            depth = len(cur.parts) - base_depth
            if depth > max_depth:
                dirnames[:] = []
                continue
            if "T1_Training" in dirnames and "T1_Validation" in dirnames:
                hits.append(cur.resolve())
                if len(hits) >= max_hits:
                    return sorted(set(hits))
    return sorted(set(hits))

roots = [Path.cwd(), Path("/content"), Path("/content/drive/MyDrive")]
found = _find_dataset_roots(roots)

print("Diagnostic search roots:")
for r in roots:
    print(" -", r)

if found:
    print("\nFound dataset root candidates:")
    for p in found:
        print(" -", p)
    print("\nSet this in Cell 4, for example:")
    print(f"DATASET_ROOT_OVERRIDE = r\"{found[0]}\"")
else:
    print("\nNo dataset root found in this runtime.")
    print("Upload/mount your Dataset first, then rerun this diagnostic and Cell 4.")

Diagnostic search roots:
 - /content
 - /content
 - /content/drive/MyDrive

Found dataset root candidates:
 - /content/dataset_from_link

Set this in Cell 4, for example:
DATASET_ROOT_OVERRIDE = r"/content/dataset_from_link"


In [11]:
def natural_key(text: str):
    return [int(c) if c.isdigit() else c for c in re.split(r"(\\d+)", text)]


def discover_nifti_files(dataset_root: Path, filename_filter: str | None = "t1") -> list[Path]:
    nii_files = sorted(dataset_root.rglob("*.nii*"), key=lambda p: natural_key(str(p).lower()))
    if filename_filter:
        k = filename_filter.lower()
        nii_files = [p for p in nii_files if k in p.name.lower()]

    if not nii_files:
        raise ValueError(
            f"No NIfTI files found under {dataset_root} with filter={filename_filter!r}"
        )
    return nii_files


def build_data_dicts(paths: list[Path]) -> list[dict]:
    return [{"image": str(p), "case_id": p.stem} for p in paths]


train_nii = discover_nifti_files(TRAIN_ROOT, filename_filter=T1_FILENAME_FILTER)
val_nii = discover_nifti_files(VAL_ROOT, filename_filter=T1_FILENAME_FILTER)
train_files = build_data_dicts(train_nii)
val_files = build_data_dicts(val_nii)

# Keep CPU fallback practical: train on a tiny subset so notebook does real work without taking hours.
if "device" in globals() and device.type != "cuda" and globals().get("CPU_FAST_MODE", False):
    max_train_cases = 8
    max_val_cases = 4
    train_files = train_files[:max_train_cases]
    val_files = val_files[:max_val_cases]
    train_nii = train_nii[:max_train_cases]
    val_nii = val_nii[:max_val_cases]
    print(f"CPU fast-mode subset active: train={len(train_files)}, val={len(val_files)}")

print(f"Train NIfTI volumes: {len(train_files)}")
print(f"Val NIfTI volumes  : {len(val_files)}")
print("Train preview:")
for p in train_nii[:3]:
    print(" -", p)
print("Validation preview:")
for p in val_nii[:3]:
    print(" -", p)

CPU fast-mode subset active: train=8, val=4
Train NIfTI volumes: 8
Val NIfTI volumes  : 4
Train preview:
 - /content/dataset_from_link/T1_Training/BraTS20_Training_001/BraTS20_Training_001_t1.nii
 - /content/dataset_from_link/T1_Training/BraTS20_Training_002/BraTS20_Training_002_t1.nii
 - /content/dataset_from_link/T1_Training/BraTS20_Training_003/BraTS20_Training_003_t1.nii
Validation preview:
 - /content/dataset_from_link/T1_Validation/BraTS20_Validation_001/BraTS20_Validation_001_t1.nii
 - /content/dataset_from_link/T1_Validation/BraTS20_Validation_002/BraTS20_Validation_002_t1.nii
 - /content/dataset_from_link/T1_Validation/BraTS20_Validation_003/BraTS20_Validation_003_t1.nii


In [12]:
train_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    EnsureTyped(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=TARGET_PIXDIM, mode=("bilinear",)),
    CropForegroundd(keys=["image"], source_key="image"),
    SpatialPadd(keys=["image"], spatial_size=TARGET_SHAPE),
    CenterSpatialCropd(keys=["image"], roi_size=TARGET_SHAPE),
    ScaleIntensityRangePercentilesd(
        keys=["image"],
        lower=NORM_LOW_PCT,
        upper=NORM_HIGH_PCT,
        b_min=-1.0,
        b_max=1.0,
        clip=True,
    ),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=0),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=1),
    RandFlipd(keys=["image"], prob=0.5, spatial_axis=2),
    RandRotate90d(keys=["image"], prob=0.3, max_k=3),
    RandAffined(
        keys=["image"],
        prob=0.15,
        rotate_range=(0.05, 0.05, 0.05),
        scale_range=(0.05, 0.05, 0.05),
        mode=("bilinear",),
        padding_mode="border",
    ),
    EnsureTyped(keys=["image"]),
])

val_transforms = Compose([
    LoadImaged(keys=["image"]),
    EnsureChannelFirstd(keys=["image"]),
    EnsureTyped(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=TARGET_PIXDIM, mode=("bilinear",)),
    CropForegroundd(keys=["image"], source_key="image"),
    SpatialPadd(keys=["image"], spatial_size=TARGET_SHAPE),
    CenterSpatialCropd(keys=["image"], roi_size=TARGET_SHAPE),
    ScaleIntensityRangePercentilesd(
        keys=["image"],
        lower=NORM_LOW_PCT,
        upper=NORM_HIGH_PCT,
        b_min=-1.0,
        b_max=1.0,
        clip=True,
    ),
    EnsureTyped(keys=["image"]),
])

train_ds = Dataset(data=train_files, transform=train_transforms)
val_ds = Dataset(data=val_files, transform=val_transforms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=(NUM_WORKERS > 0),
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=(NUM_WORKERS > 0),
)

print("Train batches:", len(train_loader))
print("Val batches  :", len(val_loader))

batch = next(iter(train_loader))
x = batch["image"]
print("Sample tensor shape:", tuple(x.shape))  # [B, 1, H, W, D]
print("Intensity min/max:", float(x.min()), float(x.max()))
if float(x.min()) < -1.05 or float(x.max()) > 1.05:
    print("Warning: intensity appears outside expected [-1, 1] range.")
else:
    print("Intensity check passed: data is in expected [-1, 1] range.")

Train batches: 8
Val batches  : 4
Sample tensor shape: (1, 1, 96, 96, 64)
Intensity min/max: -1.0 1.0
Intensity check passed: data is in expected [-1, 1] range.


/usr/local/lib/python3.12/dist-packages/monai/utils/deprecate_utils.py:321: FutureWarning: monai.transforms.spatial.dictionary Orientationd.__init__:labels: Current default value of argument `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` was changed in version None from `labels=(('L', 'R'), ('P', 'A'), ('I', 'S'))` to `labels=None`. Default value changed to None meaning that the transform now uses the 'space' of a meta-tensor, if applicable, to determine appropriate axis labels.
  warn_deprecated(argname, msg, warning_category)


In [13]:
# Rebuild DataLoaders with conservative memory settings
NUM_WORKERS = 0

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
)
val_loader = DataLoader(
    val_ds,
    batch_size=1,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=False,
    persistent_workers=False,
)

print("Safe loader override active:")
print(" - NUM_WORKERS:", NUM_WORKERS)
print(" - pin_memory:", False)

batch = next(iter(train_loader))
x = batch["image"]
print("Safe sample tensor shape:", tuple(x.shape))
print("Safe sample min/max:", float(x.min()), float(x.max()))

Safe loader override active:
 - NUM_WORKERS: 0
 - pin_memory: False
Safe sample tensor shape: (1, 1, 96, 96, 64)
Safe sample min/max: -1.0 1.0


In [24]:
# =========================
# 3D Autoencoder (T1)
# =========================
autoencoder = AutoencoderKL(
    spatial_dims=3,
    in_channels=1,
    out_channels=1,
    num_channels=(64, 128, 256),
    latent_channels=8,
    num_res_blocks=2,
    norm_num_groups=16,
    attention_levels=(False, False, True),
).to(device)

optimizer_ae = torch.optim.AdamW(autoencoder.parameters(), lr=AE_LR, weight_decay=1e-6)
scaler_ae = GradScaler(enabled=use_amp)
kl_weight = 1e-6


def kl_loss(z_mu, z_sigma):
    eps = 1e-8
    kl = 0.5 * torch.sum(
        z_mu.pow(2) + z_sigma.pow(2) - torch.log(z_sigma.pow(2) + eps) - 1,
        dim=[1, 2, 3, 4],
    )
    return kl.mean()


def _to_01(x_np: np.ndarray) -> np.ndarray:
    x_np = np.clip(x_np, -1.0, 1.0)
    return (x_np + 1.0) / 2.0


def _volume_psnr_ssim(gt_vol: np.ndarray, pred_vol: np.ndarray):
    gt = _to_01(gt_vol)
    pred = _to_01(pred_vol)

    psnr_val = psnr(gt, pred, data_range=1.0)

    ssim_slices = []
    for z in range(gt.shape[2]):
        ssim_val = ssim(gt[:, :, z], pred[:, :, z], data_range=1.0)
        ssim_slices.append(ssim_val)
    return float(psnr_val), float(np.mean(ssim_slices))


def train_autoencoder_epoch(model, loader, optimizer, scaler, device):
    model.train()
    # Re-enable gradients in case AE was previously frozen for diffusion.
    for p in model.parameters():
        p.requires_grad = True
    running = 0.0
    pbar = tqdm(loader, desc="AE Train", ncols=110)

    for step, batch in enumerate(pbar):
        t1 = batch["image"].to(device)
        optimizer.zero_grad(set_to_none=True)

        with autocast(enabled=use_amp):
            recon, z_mu, z_sigma = model(t1)
            recon_l = F.l1_loss(recon.float(), t1.float())
            loss = recon_l + kl_weight * kl_loss(z_mu, z_sigma)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running += loss.item()
        pbar.set_postfix(loss=f"{running / (step + 1):.4f}")

    return running / max(1, len(loader))


@torch.no_grad()
def validate_autoencoder_epoch(model, loader, device):
    model.eval()
    running = 0.0
    psnr_vals = []
    ssim_vals = []
    pbar = tqdm(loader, desc="AE Val", ncols=110)

    for step, batch in enumerate(pbar):
        t1 = batch["image"].to(device)
        with autocast(enabled=use_amp):
            recon, z_mu, z_sigma = model(t1)
            recon_l = F.l1_loss(recon.float(), t1.float())
            loss = recon_l + kl_weight * kl_loss(z_mu, z_sigma)

        running += loss.item()

        gt_np = t1[0, 0].detach().cpu().numpy()
        pred_np = recon[0, 0].detach().cpu().numpy()
        p_val, s_val = _volume_psnr_ssim(gt_np, pred_np)
        psnr_vals.append(p_val)
        ssim_vals.append(s_val)

        pbar.set_postfix(
            loss=f"{running / (step + 1):.4f}",
            psnr=f"{np.mean(psnr_vals):.2f}",
            ssim=f"{np.mean(ssim_vals):.4f}",
        )

    return {
        "loss": running / max(1, len(loader)),
        "psnr": float(np.mean(psnr_vals)) if psnr_vals else 0.0,
        "ssim": float(np.mean(ssim_vals)) if ssim_vals else 0.0,
    }

/tmp/ipykernel_20048/458174178.py:16: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_ae = GradScaler(enabled=use_amp)


In [25]:
# =========================
# Unconditional 3D Latent Diffusion (T1)
# =========================
latent_channels = 8

unet = DiffusionModelUNet(
    spatial_dims=3,
    in_channels=latent_channels,
    out_channels=latent_channels,
    num_res_blocks=2,
    num_channels=(128, 256, 256),
    attention_levels=(False, True, True),
    num_head_channels=(0, 64, 64),
).to(device)

ema_unet = copy.deepcopy(unet).to(device)
ema_unet.eval()
for p in ema_unet.parameters():
    p.requires_grad = False

noise_scheduler = DDPMScheduler(
    num_train_timesteps=NUM_DIFFUSION_TIMESTEPS,
    schedule="scaled_linear_beta",
    beta_start=0.00085,
    beta_end=0.012,
)

optimizer_diff = torch.optim.AdamW(unet.parameters(), lr=DIFF_LR, weight_decay=1e-6)
lr_scheduler_diff = CosineAnnealingLR(
    optimizer_diff,
    T_max=max(1, DIFF_EPOCHS),
    eta_min=DIFF_LR * 0.1,
    last_epoch=-1,
)
scaler_diff = GradScaler(enabled=use_amp)


@torch.no_grad()
def update_ema(ema_model, model, decay=0.999):
    ema_params = dict(ema_model.named_parameters())
    model_params = dict(model.named_parameters())
    for name in ema_params:
        ema_params[name].mul_(decay).add_(model_params[name], alpha=1.0 - decay)


@torch.no_grad()
def estimate_scale_factor(model, loader, device, max_batches=10):
    model.eval()
    vals = []
    for i, batch in enumerate(loader):
        t1 = batch["image"].to(device)
        z = model.encode_stage_2_inputs(t1)
        vals.append(z.std().item())
        if i + 1 >= max_batches:
            break
    return 1.0 / (np.mean(vals) + 1e-8)

scale_factor = estimate_scale_factor(autoencoder, train_loader, device)
print(f"scale_factor={scale_factor:.6f}")

/tmp/ipykernel_20048/3519320324.py:35: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler_diff = GradScaler(enabled=use_amp)


scale_factor=0.888796


In [26]:
def train_diffusion_epoch(
    autoencoder, unet, ema_unet, noise_scheduler, loader, optimizer, scaler, device,
    scale_factor, grad_accum_steps=1, ema_decay=0.999, offset_noise=0.1
):
    autoencoder.eval()
    for p in autoencoder.parameters():
        p.requires_grad = False

    unet.train()
    running = 0.0
    pbar = tqdm(loader, desc="Diff Train", ncols=120)
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(pbar):
        t1 = batch["image"].to(device)

        with torch.no_grad():
            z_t1 = autoencoder.encode_stage_2_inputs(t1) * scale_factor

        noise = torch.randn_like(z_t1)
        if offset_noise > 0:
            noise = noise + offset_noise * torch.randn(
                z_t1.shape[0], z_t1.shape[1], 1, 1, 1, device=device, dtype=z_t1.dtype
            )

        timesteps = torch.randint(
            0, noise_scheduler.num_train_timesteps, (z_t1.shape[0],), device=device, dtype=torch.long
        )

        noisy = noise_scheduler.add_noise(
            original_samples=z_t1, noise=noise, timesteps=timesteps
        )

        with autocast(enabled=use_amp):
            noise_pred = unet(noisy, timesteps=timesteps)
            if hasattr(noise_pred, "sample"):
                noise_pred = noise_pred.sample
            loss = F.mse_loss(noise_pred.float(), noise.float()) / grad_accum_steps

        scaler.scale(loss).backward()

        should_step = ((step + 1) % grad_accum_steps == 0) or (step + 1 == len(loader))
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(unet.parameters(), max_norm=1.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            update_ema(ema_unet, unet, decay=ema_decay)

        running += loss.item() * grad_accum_steps
        avg_loss = running / (step + 1)
        pbar.set_postfix(train_loss=f"{avg_loss:.4f}", lr=f"{optimizer.param_groups[0]['lr']:.2e}")

    return running / max(1, len(loader))


@torch.no_grad()
def validate_diffusion_epoch(
    autoencoder, eval_unet, noise_scheduler, loader, device, scale_factor, seed=1234
):
    autoencoder.eval()
    eval_unet.eval()
    running = 0.0
    g = torch.Generator(device=device).manual_seed(seed)

    pbar = tqdm(loader, desc="Diff Val", ncols=120)
    for step, batch in enumerate(pbar):
        t1 = batch["image"].to(device)

        z_t1 = autoencoder.encode_stage_2_inputs(t1) * scale_factor
        noise = torch.randn(z_t1.shape, generator=g, device=device, dtype=z_t1.dtype)
        timesteps = torch.randint(
            0, noise_scheduler.num_train_timesteps, (z_t1.shape[0],), generator=g, device=device
        ).long()

        noisy = noise_scheduler.add_noise(
            original_samples=z_t1, noise=noise, timesteps=timesteps
        )

        with autocast(enabled=use_amp):
            noise_pred = eval_unet(noisy, timesteps=timesteps)
            if hasattr(noise_pred, "sample"):
                noise_pred = noise_pred.sample
            loss = F.mse_loss(noise_pred.float(), noise.float())

        running += loss.item()
        pbar.set_postfix(val_loss=f"{running / (step + 1):.4f}")

    return running / max(1, len(loader))

In [27]:
@torch.no_grad()
def generate_t1(
    autoencoder,
    unet,
    scheduler,
    scale_factor,
    device,
    latent_spatial_shape,
    batch_size=1,
    latent_channels=8,
    num_inference_steps=300,
):
    autoencoder.eval()
    unet.eval()

    latent = torch.randn(
        (batch_size, latent_channels, *latent_spatial_shape),
        device=device,
    )
    scheduler.set_timesteps(num_inference_steps=num_inference_steps)

    for t in scheduler.timesteps:
        t_int = int(t.item()) if torch.is_tensor(t) else int(t)
        timesteps = torch.full((latent.shape[0],), t_int, device=device, dtype=torch.long)

        with autocast(enabled=use_amp):
            noise_pred = unet(latent, timesteps=timesteps)
            if hasattr(noise_pred, "sample"):
                noise_pred = noise_pred.sample

        step_out = scheduler.step(noise_pred, t_int, latent)
        latent = step_out[0] if isinstance(step_out, tuple) else step_out.prev_sample

    latent = latent / scale_factor
    pred_t1 = autoencoder.decode_stage_2_outputs(latent)
    pred_t1 = torch.clamp(pred_t1, -1.0, 1.0)
    return pred_t1


def save_nifti_volume(volume_tensor, save_path: Path, reference_nifti_path: str | None = None):
    vol = volume_tensor.detach().cpu().numpy()
    if vol.ndim == 4:
        vol = vol[0]
    vol = vol.astype(np.float32)

    if reference_nifti_path is not None and Path(reference_nifti_path).exists():
        ref = nib.load(reference_nifti_path)
        nii = nib.Nifti1Image(vol, affine=ref.affine, header=ref.header)
    else:
        nii = nib.Nifti1Image(vol, affine=np.eye(4))

    save_path.parent.mkdir(parents=True, exist_ok=True)
    nib.save(nii, str(save_path))

In [28]:
# =========================
# Run control
# =========================
# Adaptive execution policy:
RUN_AE_TRAIN = (device.type == "cuda") or (globals().get("CPU_FAST_MODE", False) and AE_EPOCHS > 0)
RUN_DIFF_TRAIN = device.type == "cuda"
RUN_SAMPLE = device.type == "cuda"
RUN_DIFF_PSNR_SSIM_EVAL = device.type == "cuda"
DIFF_EVAL_CASES = 1 if device.type != "cuda" else 8

if device.type != "cuda":
    if RUN_AE_TRAIN:
        print("CUDA unavailable/insufficient. Running CPU fast-mode AE training only.")
        print("Diffusion training/sampling/eval remain disabled in CPU mode.")
    else:
        print("CUDA unavailable and CPU fast-mode disabled. Training is skipped.")

scale_factor_runtime = float(scale_factor)

ckpt_dir = OUTPUT_ROOT / "checkpoints"
ckpt_dir.mkdir(parents=True, exist_ok=True)

ae_latest_path = ckpt_dir / "autoencoder_latest.pth"
ae_best_path = ckpt_dir / "autoencoder_best.pth"
diff_latest_path = ckpt_dir / "latent_diffusion_latest.pth"
diff_best_path = ckpt_dir / "latent_diffusion_best.pth"

# -------------------------
# 1) Autoencoder training
# -------------------------
best_ae_val = float("inf")
if ae_best_path.exists():
    ae_ckpt = torch.load(ae_best_path, map_location=device)
    if "autoencoder_state_dict" in ae_ckpt:
        autoencoder.load_state_dict(ae_ckpt["autoencoder_state_dict"])
        best_ae_val = ae_ckpt.get("best_val_loss", best_ae_val)
    print(f"Loaded AE checkpoint: {ae_best_path}")

if RUN_AE_TRAIN:
    for epoch in range(AE_EPOCHS):
        tr = train_autoencoder_epoch(autoencoder, train_loader, optimizer_ae, scaler_ae, device)
        va = validate_autoencoder_epoch(autoencoder, val_loader, device)
        print(
            f"AE Epoch {epoch + 1}/{AE_EPOCHS} | train_loss={tr:.6f} | val_loss={va['loss']:.6f} | val_psnr={va['psnr']:.3f} | val_ssim={va['ssim']:.4f}"
        )

        ae_state = {
            "epoch": epoch + 1,
            "autoencoder_state_dict": autoencoder.state_dict(),
            "optimizer_state_dict": optimizer_ae.state_dict(),
            "best_val_loss": best_ae_val,
            "last_val_psnr": va["psnr"],
            "last_val_ssim": va["ssim"],
            "config": {
                "target_shape": TARGET_SHAPE,
                "target_pixdim": TARGET_PIXDIM,
            },
        }
        torch.save(ae_state, ae_latest_path)

        if va["loss"] < best_ae_val:
            best_ae_val = va["loss"]
            ae_state["best_val_loss"] = best_ae_val
            torch.save(ae_state, ae_best_path)
            print(f"Saved new best AE checkpoint (val_loss={best_ae_val:.6f})")

# Always load best AE before diffusion
if ae_best_path.exists():
    ae_ckpt = torch.load(ae_best_path, map_location=device)
    autoencoder.load_state_dict(ae_ckpt["autoencoder_state_dict"])
autoencoder.eval()
for p in autoencoder.parameters():
    p.requires_grad = False

# -------------------------
# 2) Diffusion training
# -------------------------
best_diff_val = float("inf")
if diff_best_path.exists():
    d_ckpt = torch.load(diff_best_path, map_location=device)
    if "unet_state_dict" in d_ckpt:
        unet.load_state_dict(d_ckpt["unet_state_dict"])
    if "ema_unet_state_dict" in d_ckpt:
        ema_unet.load_state_dict(d_ckpt["ema_unet_state_dict"])
    scale_factor_runtime = float(d_ckpt.get("scale_factor", scale_factor_runtime))
    best_diff_val = d_ckpt.get("best_val_loss", best_diff_val)
    print(f"Loaded diffusion checkpoint: {diff_best_path}")
    print(f"Restored scale_factor={scale_factor_runtime:.6f}")

if RUN_DIFF_TRAIN:
    scale_factor_runtime = float(estimate_scale_factor(autoencoder, train_loader, device))
    print(f"Estimated scale_factor={scale_factor_runtime:.6f}")

    for epoch in range(DIFF_EPOCHS):
        tr = train_diffusion_epoch(
            autoencoder=autoencoder,
            unet=unet,
            ema_unet=ema_unet,
            noise_scheduler=noise_scheduler,
            loader=train_loader,
            optimizer=optimizer_diff,
            scaler=scaler_diff,
            device=device,
            scale_factor=scale_factor_runtime,
            grad_accum_steps=GRAD_ACCUM_STEPS,
            ema_decay=EMA_DECAY,
            offset_noise=OFFSET_NOISE,
        )

        va = validate_diffusion_epoch(
            autoencoder=autoencoder,
            eval_unet=ema_unet,
            noise_scheduler=noise_scheduler,
            loader=val_loader,
            device=device,
            scale_factor=scale_factor_runtime,
            seed=1234 + epoch,
        )

        lr_scheduler_diff.step()
        print(
            f"Diff Epoch {epoch + 1}/{DIFF_EPOCHS} | train={tr:.6f} | val_noise_mse={va:.6f} | lr={optimizer_diff.param_groups[0]['lr']:.2e}"
        )

        d_state = {
            "epoch": epoch + 1,
            "unet_state_dict": unet.state_dict(),
            "ema_unet_state_dict": ema_unet.state_dict(),
            "optimizer_state_dict": optimizer_diff.state_dict(),
            "scaler_state_dict": scaler_diff.state_dict(),
            "best_val_loss": best_diff_val,
            "scale_factor": scale_factor_runtime,
            "latent_channels": latent_channels,
            "config": {
                "target_shape": TARGET_SHAPE,
                "target_pixdim": TARGET_PIXDIM,
                "num_diffusion_timesteps": NUM_DIFFUSION_TIMESTEPS,
            },
        }
        torch.save(d_state, diff_latest_path)

        if va < best_diff_val:
            best_diff_val = va
            d_state["best_val_loss"] = best_diff_val
            torch.save(d_state, diff_best_path)
            print(f"Saved new best diffusion checkpoint (val_noise_mse={best_diff_val:.6f})")

        if (epoch + 1) % SAVE_EVERY == 0:
            torch.save(d_state, ckpt_dir / f"latent_diffusion_epoch_{epoch + 1:04d}.pth")

# -------------------------
# 3) Sampling
# -------------------------
if RUN_SAMPLE:
    if diff_best_path.exists():
        d_ckpt = torch.load(diff_best_path, map_location=device)
        if "unet_state_dict" in d_ckpt:
            unet.load_state_dict(d_ckpt["unet_state_dict"])
        if "ema_unet_state_dict" in d_ckpt:
            ema_unet.load_state_dict(d_ckpt["ema_unet_state_dict"])
        scale_factor_runtime = float(d_ckpt.get("scale_factor", scale_factor_runtime))
    else:
        print("Warning: diffusion best checkpoint not found, sampling with current in-memory weights.")

    sample_batch = next(iter(val_loader))
    sample_real = sample_batch["image"].to(device)
    with torch.no_grad():
        z_shape = autoencoder.encode_stage_2_inputs(sample_real[:1]).shape[2:]

    pred_t1 = generate_t1(
        autoencoder=autoencoder,
        unet=ema_unet,
        scheduler=noise_scheduler,
        scale_factor=scale_factor_runtime,
        device=device,
        latent_spatial_shape=z_shape,
        batch_size=1,
        latent_channels=latent_channels,
        num_inference_steps=INFERENCE_STEPS,
    )

    pred_path = OUTPUT_ROOT / "generated" / "sample_synth_t1.nii.gz"
    ref_path = val_files[0]["image"] if len(val_files) > 0 else None
    save_nifti_volume(pred_t1[0], pred_path, reference_nifti_path=ref_path)
    print(f"Saved synthetic sample: {pred_path}")

# -------------------------
# 4) Diffusion PSNR/SSIM evaluation (optional)
# -------------------------
if RUN_DIFF_PSNR_SSIM_EVAL:
    if diff_best_path.exists():
        d_ckpt = torch.load(diff_best_path, map_location=device)
        if "ema_unet_state_dict" in d_ckpt:
            ema_unet.load_state_dict(d_ckpt["ema_unet_state_dict"])
        scale_factor_runtime = float(d_ckpt.get("scale_factor", scale_factor_runtime))

    max_cases = min(DIFF_EVAL_CASES, len(val_loader))
    psnr_vals = []
    ssim_vals = []

    for i, batch in enumerate(val_loader):
        if i >= max_cases:
            break
        real_t1 = batch["image"].to(device)
        with torch.no_grad():
            z_shape = autoencoder.encode_stage_2_inputs(real_t1[:1]).shape[2:]

        fake_t1 = generate_t1(
            autoencoder=autoencoder,
            unet=ema_unet,
            scheduler=noise_scheduler,
            scale_factor=scale_factor_runtime,
            device=device,
            latent_spatial_shape=z_shape,
            batch_size=1,
            latent_channels=latent_channels,
            num_inference_steps=INFERENCE_STEPS,
        )

        gt_np = real_t1[0, 0].detach().cpu().numpy()
        pred_np = fake_t1[0, 0].detach().cpu().numpy()
        p_val, s_val = _volume_psnr_ssim(gt_np, pred_np)
        psnr_vals.append(p_val)
        ssim_vals.append(s_val)
        print(f"Eval case {i+1}/{max_cases} | PSNR={p_val:.3f} | SSIM={s_val:.4f}")

    if psnr_vals:
        print("===== Diffusion Evaluation (against matched validation volumes) =====")
        print(f"Mean PSNR: {np.mean(psnr_vals):.3f}")
        print(f"Mean SSIM: {np.mean(ssim_vals):.4f}")
        print("Note: for unconditional generation, this is only a rough proxy metric.")

CUDA unavailable/insufficient. Running CPU fast-mode AE training only.
Diffusion training/sampling/eval remain disabled in CPU mode.
Loaded AE checkpoint: /content/outputs/synthetic_t1_3d_ldm/checkpoints/autoencoder_best.pth


AE Train:   0%|                                                                         | 0/8 [00:00<?, ?it/s]

/tmp/ipykernel_20048/458174178.py:59: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


AE Val:   0%|                                                                           | 0/4 [00:00<?, ?it/s]

/tmp/ipykernel_20048/458174178.py:84: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(enabled=use_amp):


AE Epoch 1/1 | train_loss=0.442586 | val_loss=0.323900 | val_psnr=13.835 | val_ssim=0.2628
Saved new best AE checkpoint (val_loss=0.323900)


In [29]:
# =========================
# Final export: package model artifacts and download locally
# =========================
import zipfile
from datetime import datetime
from pathlib import Path

EXPORT_BASENAME = "MDS03_Backend_synthetic_t1_3d_ldm_artifacts"
EXPORT_LOCAL_DIR = Path("/content")
EXPORT_DRIVE_DIR = Path("/content/drive/MyDrive/MDS03_Backend_exports")
TRIGGER_BROWSER_DOWNLOAD = True
COPY_ARCHIVE_TO_DRIVE = True
INCLUDE_ALL_EPOCH_CHECKPOINTS = False

runtime_output_root = OUTPUT_ROOT if "OUTPUT_ROOT" in globals() else Path("/content/outputs/synthetic_t1_3d_ldm")
runtime_ckpt_dir = runtime_output_root / "checkpoints"
runtime_generated_dir = runtime_output_root / "generated"

artifact_files = []

# Priority checkpoints to include
priority_ckpt_names = [
    "autoencoder_best.pth",
    "autoencoder_latest.pth",
    "latent_diffusion_best.pth",
    "latent_diffusion_latest.pth",
]
for name in priority_ckpt_names:
    p = runtime_ckpt_dir / name
    if p.exists():
        artifact_files.append(p)

# Optional full checkpoint history (can be very large)
if INCLUDE_ALL_EPOCH_CHECKPOINTS and runtime_ckpt_dir.exists():
    artifact_files.extend(sorted(runtime_ckpt_dir.glob("latent_diffusion_epoch_*.pth")))

# Include generated samples if present
if runtime_generated_dir.exists():
    artifact_files.extend(sorted(runtime_generated_dir.glob("*.nii*")))

# De-duplicate while preserving order
seen = set()
deduped = []
for p in artifact_files:
    s = str(p)
    if s not in seen:
        seen.add(s)
        deduped.append(p)
artifact_files = deduped

if not artifact_files:
    print("No model artifacts found yet.")
    print("Run training/sampling first, then rerun this export cell.")
else:
    EXPORT_LOCAL_DIR.mkdir(parents=True, exist_ok=True)
    ts = datetime.now().strftime("%Y%m%d_%H%M%S")
    archive_path = EXPORT_LOCAL_DIR / f"{EXPORT_BASENAME}_{ts}.zip"

    # Write manifest for traceability
    manifest_path = EXPORT_LOCAL_DIR / f"{EXPORT_BASENAME}_{ts}_manifest.txt"
    with open(manifest_path, "w", encoding="utf-8") as f:
        f.write("Exported artifact files:\n")
        for p in artifact_files:
            f.write(f"- {p}\n")

    with zipfile.ZipFile(archive_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
        for p in artifact_files:
            if p.exists():
                try:
                    arcname = p.relative_to(runtime_output_root.parent)
                except Exception:
                    arcname = p.name
                zf.write(p, arcname=str(arcname))
        zf.write(manifest_path, arcname=manifest_path.name)

    print(f"Created archive: {archive_path}")

    if COPY_ARCHIVE_TO_DRIVE and Path("/content/drive/MyDrive").exists():
        EXPORT_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
        drive_archive_path = EXPORT_DRIVE_DIR / archive_path.name
        import shutil
        shutil.copy2(archive_path, drive_archive_path)
        print(f"Copied archive to Drive: {drive_archive_path}")

    if TRIGGER_BROWSER_DOWNLOAD:
        try:
            from google.colab import files
            print("Triggering browser download ...")
            files.download(str(archive_path))
            print("After download completes, move the ZIP into your local MDS03_Backend folder.")
        except Exception as e:
            print(f"Browser download not triggered automatically: {e}")
            print(f"Manually download this file: {archive_path}")

Created archive: /content/MDS03_Backend_synthetic_t1_3d_ldm_artifacts_20260326_172659.zip
Copied archive to Drive: /content/drive/MyDrive/MDS03_Backend_exports/MDS03_Backend_synthetic_t1_3d_ldm_artifacts_20260326_172659.zip
Triggering browser download ...


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

After download completes, move the ZIP into your local MDS03_Backend folder.
